In [3]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display
from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
import gradio as gr

warnings.filterwarnings('ignore')

PROJECT_ROOT   = Path('data')
AUDIO_ROOT     = PROJECT_ROOT / 'audio'
CSV_PATH       = PROJECT_ROOT / 'real_dialect_mfccs.csv'
GEOSPATIAL_CSV = PROJECT_ROOT / 'abandonment_risk_proxies.csv'

LANGS = {'hindi': 'Hindi', 'tamil': 'Tamil', 'bengali': 'Bengali'}

for lang in LANGS:
    (AUDIO_ROOT / lang).mkdir(parents=True, exist_ok=True)

print(f"📁 Project root: {PROJECT_ROOT.resolve()}")

📁 Project root: /content/data


In [5]:
def run_geospatial_module():
    print("\n" + "="*60)
    print("MODULE 1: Geospatial Abandonment Risk Mapping")
    print("="*60)

    if not GEOSPATIAL_CSV.exists():
        print(f"⚠️  Geospatial CSV not found at: {GEOSPATIAL_CSV}")
        return

    df = pd.read_csv(GEOSPATIAL_CSV)
    features = ['Latitude', 'Longitude', 'MPI_Headcount_Ratio',
                'Elder_Abuse_Prevalence', 'Transit_Hub_Score']
    X = df[features]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    wcss = []
    for i in range(1, 6):
        km = KMeans(n_clusters=i, init='k-means++', max_iter=300,
                    n_init=10, random_state=42)
        km.fit(X_scaled)
        wcss.append(km.inertia_)

    plt.figure(figsize=(8, 5))
    plt.plot(range(1, 6), wcss, marker='o', linestyle='--')
    plt.title('Elbow Method (Determining Optimal k)')
    plt.xlabel('Number of Clusters (k)')
    plt.ylabel('WCSS / Inertia')
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    k = 3
    kmeans = KMeans(n_clusters=k, init='k-means++', max_iter=300,
                    n_init=10, random_state=42)
    df['Hotspot_Cluster'] = kmeans.fit_predict(X_scaled)

    cluster_means = df.groupby('Hotspot_Cluster')[
        ['MPI_Headcount_Ratio', 'Elder_Abuse_Prevalence', 'Transit_Hub_Score']
    ].mean()
    print("\n--- Cluster Mean Analysis ---")
    print(cluster_means)

    risk_score = cluster_means['MPI_Headcount_Ratio'] + cluster_means['Transit_Hub_Score']
    sorted_clusters = risk_score.sort_values(ascending=False).index
    risk_map = {
        sorted_clusters[0]: 'High Risk',
        sorted_clusters[1]: 'Medium Risk',
        sorted_clusters[2]: 'Low Risk'
    }
    df['Risk_Level'] = df['Hotspot_Cluster'].map(risk_map)

    print("\n--- Predictive Abandonment Hotspot Results (Tamil Nadu Districts) ---")
    print(df[['District_ID', 'Latitude', 'Longitude', 'Risk_Level']])
    print("\n💡 Recommendation: Focus SSA Samanvay Setu pilot resources on 'High Risk' districts.")

    plt.figure(figsize=(10, 6))
    scatter = plt.scatter(
        df['Longitude'], df['Latitude'],
        c=df['Hotspot_Cluster'], cmap='RdYlGn_r',
        s=df['Transit_Hub_Score'] * 400, alpha=0.8, edgecolors='k'
    )
    plt.scatter(
        kmeans.cluster_centers_[:, 1],
        kmeans.cluster_centers_[:, 0],
        s=350, c='blue', marker='X', label='Centroids'
    )
    plt.title(f'Geospatial Abandonment Risk Mapping (k={k})')
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.legend()
    plt.colorbar(scatter, label='Cluster ID')
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    return df

In [6]:
def extract_audio_features(audio_path, sr=22050, n_mfcc=13):
    try:
        y, sr = librosa.load(audio_path, sr=sr, mono=True)
        y, _  = librosa.effects.trim(y)

        mfccs        = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
        mfccs_mean   = np.mean(mfccs.T, axis=0)
        delta_mfccs  = librosa.feature.delta(mfccs)
        delta_mean   = np.mean(delta_mfccs.T, axis=0)
        delta2_mfccs = librosa.feature.delta(mfccs, order=2)
        delta2_mean  = np.mean(delta2_mfccs.T, axis=0)

        f0, voiced_flag, _ = librosa.pyin(
            y, fmin=librosa.note_to_hz('C2'),
            fmax=librosa.note_to_hz('C6'), sr=sr
        )
        f0_voiced = f0[voiced_flag] if voiced_flag is not None else np.array([])
        f0_mean   = float(np.mean(f0_voiced))   if len(f0_voiced) > 0 else 0.0
        f0_std    = float(np.std(f0_voiced))    if len(f0_voiced) > 0 else 0.0

        rms_mean = float(np.mean(librosa.feature.rms(y=y)))

        contrast      = librosa.feature.spectral_contrast(y=y, sr=sr)
        contrast_mean = np.mean(contrast.T, axis=0)

        chroma      = librosa.feature.chroma_stft(y=y, sr=sr)
        chroma_mean = np.mean(chroma.T, axis=0)

        feat = {f'mfcc_{i+1}':      mfccs_mean[i]   for i in range(n_mfcc)}
        feat.update({f'delta_mfcc_{i+1}':  delta_mean[i]   for i in range(n_mfcc)})
        feat.update({f'delta2_mfcc_{i+1}': delta2_mean[i]  for i in range(n_mfcc)})
        feat.update({f'contrast_{i+1}':    contrast_mean[i] for i in range(len(contrast_mean))})
        feat.update({f'chroma_{i+1}':      chroma_mean[i]   for i in range(len(chroma_mean))})
        feat.update({'pitch_mean': f0_mean, 'pitch_std': f0_std, 'rms_mean': rms_mean})

        return feat, mfccs

    except Exception as e:
        print(f"❌ Error processing {audio_path}: {e}")
        return None, np.array([])


def display_mfcc(mfccs, title="MFCC Spectrogram"):
    if mfccs.size > 0:
        plt.figure(figsize=(10, 4))
        librosa.display.specshow(mfccs, x_axis='time')
        plt.colorbar()
        plt.title(title)
        plt.tight_layout()
        plt.show()

In [7]:
def train_dialect_model():
    print("\n" + "="*60)
    print("MODULE 3: Dialect Model Training")
    print("="*60)

    dataset = []

    for folder, label in LANGS.items():
        lang_path = AUDIO_ROOT / folder
        audio_files = (
            list(lang_path.glob('*.wav')) +
            list(lang_path.glob('*.mp3')) +
            list(lang_path.glob('*.m4a'))
        )
        print(f"🔍 {label}: Found {len(audio_files)} file(s).")

        for f_path in audio_files:
            feat, _ = extract_audio_features(str(f_path))
            if feat:
                feat['Language_Label'] = label
                feat['File_Name']      = f_path.name
                dataset.append(feat)

    if not dataset:
        print("⚠️  No audio files found. Add files into the data/audio subfolders first.")
        return None, None

    df = pd.DataFrame(dataset)
    df.to_csv(CSV_PATH, index=False)
    print(f"✅ Dataset saved → {CSV_PATH}  ({len(df)} samples)")

    return _fit_model(df)


def _fit_model(df):
    feature_cols = [c for c in df.columns
                    if c not in ('Language_Label', 'File_Name')]
    X = df[feature_cols].values
    y = df['Language_Label'].values

    n_samples = len(X)
    if n_samples >= 30:
        clf = SVC(kernel='rbf', C=10, gamma='scale',
                  probability=True, random_state=42)
        model_name = "SVM (RBF)"
    else:
        clf = KNeighborsClassifier(n_neighbors=3, metric='euclidean',
                                   weights='distance')
        model_name = "KNN (k=3, distance-weighted)"

    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    clf)
    ])

    if n_samples >= 10:
        cv      = StratifiedKFold(n_splits=min(5, n_samples // len(LANGS)),
                                  shuffle=True, random_state=42)
        cv_acc  = cross_val_score(pipe, X, y, cv=cv, scoring='accuracy')
        print(f"\n📊 Cross-Validation Accuracy ({model_name}): "
              f"{cv_acc.mean()*100:.1f}% ± {cv_acc.std()*100:.1f}%")

        if n_samples >= 20:
            X_tr, X_te, y_tr, y_te = train_test_split(
                X, y, test_size=0.2, stratify=y, random_state=42
            )
            pipe.fit(X_tr, y_tr)
            y_pred = pipe.predict(X_te)
            print("\n--- Classification Report ---")
            print(classification_report(y_te, y_pred))
    else:
        print(f"⚠️  Only {n_samples} samples — skipping cross-val. Add more audio for reliable metrics.")

    pipe.fit(X, y)
    print(f"✅ Model ({model_name}) fitted on {n_samples} samples.")
    return pipe, feature_cols


def load_model_from_csv():
    if not CSV_PATH.exists():
        print(f"❌ CSV not found at {CSV_PATH}. Run train_dialect_model() first.")
        return None, None
    df = pd.read_csv(CSV_PATH)
    print(f"✅ Loaded existing dataset: {len(df)} samples from {CSV_PATH}")
    return _fit_model(df)

In [8]:
def perform_triage(pitch_std: float, rms_mean: float):
    if rms_mean < 0.01:
        return ("🔴 HIGH PRIORITY — Weak / Non-responsive",
                "Urgent: Refer for immediate medical evaluation.")
    elif pitch_std < 45:
        return ("🟡 MEDIUM PRIORITY — Subdued / Possible Depression",
                "Recommended: Psychological evaluation and social support.")
    else:
        return ("🟢 STABLE",
                "Standard care protocol applies.")


def analyze_senior_voice(audio_file, pipe, feature_cols):
    if audio_file is None:
        return "No file provided", "", ""
    if pipe is None:
        return "Model not trained", "", ""

    feat, mfccs = extract_audio_features(audio_file)
    if feat is None:
        return "Feature extraction failed", "", ""

    display_mfcc(mfccs, title=f"MFCC — {Path(audio_file).name}")

    test_vec = np.array([[feat.get(c, 0.0) for c in feature_cols]])

    dialect = pipe.predict(test_vec)[0]
    probs   = pipe.predict_proba(test_vec)[0]
    classes = pipe.classes_

    prob_df = pd.DataFrame({'Dialect': classes, 'Probability': probs})
    prob_df = prob_df.sort_values('Probability', ascending=False)

    print("\n--- 🧠 Geo-Linguistic Profiling ---")
    print(prob_df.head(3).to_string(index=False))

    status, recommendation = perform_triage(feat['pitch_std'], feat['rms_mean'])

    print(f"\n🎯 Predicted Dialect : {dialect}")
    print(f"🩺 Triage Status     : {status}")
    print(f"📋 Recommendation    : {recommendation}")

    return dialect, status, recommendation


def debug_single_file(audio_path: str, pipe, feature_cols):
    print(f"\n🔬 Debugging: {Path(audio_path).name}")

    feat, _ = extract_audio_features(audio_path)
    if feat is None:
        return

    test_vec  = np.array([[feat.get(c, 0.0) for c in feature_cols]])
    scaler    = pipe.named_steps['scaler']
    clf       = pipe.named_steps['clf']
    X_scaled  = scaler.transform(test_vec)

    probs   = clf.predict_proba(X_scaled)[0]
    classes = clf.classes_

    print("\n--- Confidence Report ---")
    for cls, prob in zip(classes, probs):
        tag = "✅" if prob > 0.5 else "⚠️"
        print(f"{tag}  {cls:<20}: {prob*100:6.1f}%")
    print(f"\n🎯 Final Decision: {clf.predict(X_scaled)[0]}")

In [9]:
def launch_ui(pipe, feature_cols):
    print("\n" + "="*60)
    print("MODULE 6: Launching Gradio UI")
    print("="*60)

    def ui_handler(audio_file):
        return analyze_senior_voice(audio_file, pipe, feature_cols)

    with gr.Blocks(theme=gr.themes.Soft()) as demo:
        gr.Markdown("# 🏠 Sahara Sevak AI — Samanvay Setu")
        gr.Markdown("### Intake Portal for Abandoned Senior Citizens")

        with gr.Row():
            with gr.Column():
                audio_input = gr.Audio(
                    type="filepath",
                    label="Record or Upload Senior's Voice"
                )
                submit_btn  = gr.Button("Analyze Voice Data", variant="primary")

            with gr.Column():
                out_dialect = gr.Label(label="Probable Regional Origin (Dialect)")
                out_status  = gr.Textbox(label="Health & Emotional Triage Status")
                out_rec     = gr.Textbox(label="System Recommendation")

        submit_btn.click(
            fn=ui_handler,
            inputs=audio_input,
            outputs=[out_dialect, out_status, out_rec]
        )

    demo.launch(share=True)

In [10]:
run_geospatial_module()

if CSV_PATH.exists():
    pipe, feature_cols = load_model_from_csv()
else:
    pipe, feature_cols = train_dialect_model()

# Optional: test a specific file
# debug_single_file('data/audio/tamil/audio 4 T.wav', pipe, feature_cols)

if pipe is not None:
    launch_ui(pipe, feature_cols)
else:
    print("\n⚠️  No model available. Add audio files to the subfolders and re-run.")


MODULE 1: Geospatial Abandonment Risk Mapping
⚠️  Geospatial CSV not found at: data/abandonment_risk_proxies.csv

MODULE 3: Dialect Model Training
🔍 Hindi: Found 0 file(s).
🔍 Tamil: Found 0 file(s).
🔍 Bengali: Found 0 file(s).
⚠️  No audio files found. Add files into the data/audio subfolders first.

⚠️  No model available. Add audio files to the subfolders and re-run.
